# 06 — Graph IO: GraphML Round-Trip and Interoperability

**Goal:** Build a small graph, save it to GraphML, read it back, verify
the round-trip, and understand the tensor-feature limitations.

**TGraphX subsystem:** `tgraphx.io`

**Data:** Synthetic — no download.

**Runtime:** < 10 seconds on CPU.

In [ ]:
import torch
from pathlib import Path
import tempfile
from tgraphx import Graph
from tgraphx.io import write_graphml, read_graphml
print("tgraphx.io ready")

## 2. Build a Small Graph with Metadata

In [ ]:
# Create a directed 5-node graph with edge weights and labels.
x = torch.tensor([          # Scalar node features [5, 1] (1-D → round-trips)
    [0.1], [0.5], [0.9], [0.3], [0.7]
])
edge_index = torch.tensor([
    [0, 1, 2, 2, 3, 4],    # source nodes
    [1, 2, 3, 4, 4, 0],    # target nodes
], dtype=torch.long)
edge_weight = torch.tensor([1.5, 2.0, 0.5, 1.0, 3.0, 2.5])
y = torch.tensor([0, 1, 1, 0, 2], dtype=torch.long)  # node labels

g = Graph(node_features=x, edge_index=edge_index, edge_weight=edge_weight, y=y)
print(g)

## 3. Write to GraphML and Read Back

In [ ]:
with tempfile.NamedTemporaryFile(suffix=".graphml", delete=False) as f:
    path = Path(f.name)

# Write
write_graphml(g, path, include_labels=True, include_tensor_features=True)
print("Written:", path)
print("File size:", path.stat().st_size, "bytes")

# Read
g2 = read_graphml(path, feature_dtype=torch.float32)
print("\nRound-trip result:")
print(f"  Nodes: {g2.num_nodes} (expected {g.num_nodes})")
print(f"  Edges: {g2.num_edges} (expected {g.num_edges})")
print(f"  node_features: {g2.node_features}")
print(f"  node_labels: {g2.node_labels}")
print(f"  edge_weight: {g2.edge_weight}")

# Clean up
path.unlink()

## 4. What Does NOT Round-Trip (and Why)

In [ ]:
# Attempt to save [N, C, H, W] tensor features — TGraphX refuses safely.
x_spatial = torch.randn(4, 3, 8, 8)   # image-like node features
g_spatial = Graph(node_features=x_spatial,
                  edge_index=torch.tensor([[0,1],[1,2]]))
try:
    with tempfile.NamedTemporaryFile(suffix=".graphml", delete=False) as f:
        write_graphml(g_spatial, f.name, include_tensor_features=True)
except ValueError as e:
    print("ValueError (expected):", e)
print("\nTGraphX refuses to silently flatten [N,C,H,W] through GraphML.")
print("For lossless persistence: use torch.save({'graph': g_spatial}, 'graph.pt')")

## 5. Interoperability

GraphML files written by TGraphX can be opened in:
- **NetworkX**: `import networkx as nx; G = nx.read_graphml("out.graphml")`
- **Gephi**: File → Open
- **Cytoscape**: File → Import → Network from File

Node labels and edge weights survive the export (as XML attributes).

**What GraphML cannot express:**
- `[C, H, W]` tensor node features
- `graph_features` (graph-level tensors)
- arbitrary metadata dicts

For these, use `torch.save` or write a custom JSON+tensor pair.

## 6. Next Steps
- **IO docs:** `docs/io.md`
- **Roadmap:** GEXF and Pajek planned for v1.4